FIRST ADJUSTMENTS


In [36]:
import pandas as pd
# ===============================
# 2. CARREGAR DATASETS 
# ===============================
glucose = pd.read_csv('minha_pasta/sharpic-ManchesterCSCoordinatedDiabetesStudy-fdbd74f/Glucose Data/UoMGlucose2301.csv')
meals = pd.read_csv('minha_pasta/sharpic-ManchesterCSCoordinatedDiabetesStudy-fdbd74f/Nutrition Data/UoMNutrition2301.csv')
bolus = pd.read_csv('minha_pasta/sharpic-ManchesterCSCoordinatedDiabetesStudy-fdbd74f/Insulin Data/Bolus Data/UoMBolus2301.csv')

# Renomear colunas de tempo para o formato comum "timestamp"
glucose.rename(columns={'bg_ts': 'timestamp'}, inplace=True)
meals.rename(columns={'meal_ts': 'timestamp'}, inplace=True)
bolus.rename(columns={'bolus_ts': 'timestamp'}, inplace=True)

# Converter para datetime (formato DD/MM/YYYY)
glucose["timestamp"] = pd.to_datetime(glucose["timestamp"], dayfirst=True, errors='coerce')
meals["timestamp"] = pd.to_datetime(meals["timestamp"], dayfirst=True, errors='coerce')
bolus["timestamp"] = pd.to_datetime(bolus["timestamp"], dayfirst=True, errors='coerce')


FIRST PIPELINE - IT IS THE BASE FOR THE OTHER ONES. DO NOT TOUCH.


In [29]:
import os
import pandas as pd
import matplotlib.pyplot as plt
from datetime import timedelta

# ===============================
# 1. CRIAR DIRETORIAS
# ===============================
base_dir = "data"
dirs = ["processed"]
for d in dirs:
    os.makedirs(os.path.join(base_dir, d), exist_ok=True)

# ===============================
# 2. CARREGAR DATASETS 
# ===============================

import pandas as pd

glucose = pd.read_csv('minha_pasta/sharpic-ManchesterCSCoordinatedDiabetesStudy-fdbd74f/Glucose Data/UoMGlucose2301.csv')
meals   = pd.read_csv('minha_pasta/sharpic-ManchesterCSCoordinatedDiabetesStudy-fdbd74f/Nutrition Data/UoMNutrition2301.csv')
bolus   = pd.read_csv('minha_pasta/sharpic-ManchesterCSCoordinatedDiabetesStudy-fdbd74f/Insulin Data/Bolus Data/UoMBolus2301.csv')

# Renomear colunas de tempo para um nome comum
glucose.rename(columns={'bg_ts': 'timestamp'}, inplace=True)
meals.rename(columns={'meal_ts': 'timestamp'}, inplace=True)
bolus.rename(columns={'bolus_ts': 'timestamp'}, inplace=True)

# Converter para datetime (formato DD/MM/YYYY)
glucose["timestamp"] = pd.to_datetime(glucose["timestamp"], dayfirst=True, errors='coerce')
meals["timestamp"] = pd.to_datetime(meals["timestamp"], dayfirst=True, errors='coerce')
bolus["timestamp"] = pd.to_datetime(bolus["timestamp"], dayfirst=True, errors='coerce')

# Validação opcional — ver se há datas inválidas
for name, df in [("value", glucose), ("meal_type", meals), ("bolus_dose", bolus)]:
    invalid = df["timestamp"].isna().sum()
    if invalid > 0:
        print(f"⚠️ {invalid} timestamps inválidos encontrados em {name} (convertidos em NaT)")


# ===============================
# 2.5 CRIAR IDS ÚNICOS
# ===============================

glucose["id"] = range(1, len(glucose) + 1)
meals["id"]   = range(1, len(meals) + 1)
bolus["id"]   = range(1, len(bolus) + 1)


# ===============================
# 3. FUNÇÕES AUXILIARES
# ===============================

def get_next_meal(current_meal, all_meals):
    """Devolve a refeição seguinte."""
    later_meals = all_meals[all_meals["timestamp"] > current_meal["timestamp"]]
    if later_meals.empty:
        return None
    return later_meals.iloc[0]

def find_peak(glucose_window):
    """Encontra o pico de glicose num intervalo."""
    idx = glucose_window["value"].idxmax()
    return glucose_window.loc[idx, "timestamp"], glucose_window.loc[idx, "value"]

def find_recovery(glucose_window, baseline):
    """Encontra o momento em que a glicose volta ao baseline."""
    post_meal = glucose_window[glucose_window["value"] <= baseline]
    if post_meal.empty:
        return None
    return post_meal.iloc[0]["timestamp"]

# ===============================
# 4. CRIAR EVENTOS COMPORTAMENTAIS
# ===============================
events = []

for _, meal in meals.iterrows():
    start_time = meal["timestamp"]
    end_time = start_time + timedelta(hours=4)
    
    next_meal = get_next_meal(meal, meals)
    if next_meal is not None and next_meal["timestamp"] < end_time:
        end_time = next_meal["timestamp"]
    
    glucose_window = glucose[(glucose["timestamp"] >= start_time) & (glucose["timestamp"] <= end_time)]
    if glucose_window.empty:
        continue

    # Baseline = média 30 min antes da refeição
    glucose_before = glucose[(glucose["timestamp"] >= start_time - timedelta(minutes=30)) &
                             (glucose["timestamp"] < start_time)]
    baseline = glucose_before["value"].mean() if not glucose_before.empty else glucose_window.iloc[0]["value"]

    # Evento: Meal
    events.append({
        "case_id": meal["id"],
        "event_type": "Meal",
        "timestamp": start_time,
        "value": None
    })

    # Evento: Bolus (se aplicável)
    bolus_for_meal = bolus[(bolus["timestamp"] >= start_time - timedelta(minutes=15)) &
                           (bolus["timestamp"] <= start_time + timedelta(minutes=15))]
    for _, b in bolus_for_meal.iterrows():
        events.append({
            "case_id": meal["id"],
            "event_type": "BolusInsulin",
            "timestamp": b["timestamp"],
            "value": b.get("bolus_dose", None)
        })

    # Evento: Pico de glicose
    peak_time, peak_value = find_peak(glucose_window)
    events.append({
        "case_id": meal["id"],
        "event_type": "PPGR_peak",
        "timestamp": peak_time,
        "value": peak_value
    })

    # Evento: Retorno ao normal
   recovery_time = find_recovery(glucose_window, baseline)
    if recovery_time:
        events.append({
            "case_id": meal["id"],
            "event_type": "PPGR_recovery",
            "timestamp": recovery_time,
            "value": baseline
        })


    # ===============================
    # 5. GERAR GRÁFICO
    # ===============================
    plt.figure(figsize=(10,5))
    plt.plot(glucose_window["timestamp"], glucose_window["value"], '-o', label="Glucose")
    plt.axvline(start_time, color='green', linestyle='--', label='Meal')
    plt.scatter(peak_time, peak_value, color='red', marker='^', label='Peak')
    if recovery_time:
        plt.axvline(recovery_time, color='orange', linestyle='--', label='Return to baseline')
    plt.axhline(baseline, color='gray', linestyle=':', label=f'Baseline ({baseline:.1f})')
    plt.xlabel("Time")
    plt.ylabel("Glucose (mmol/L)")
    plt.title(f"Postprandial Glucose Response - Meal {meal['id']}")
    plt.legend()
    plt.tight_layout()
    plt.savefig(f"data/processed/meal_{meal['id']}_curve.png")
    plt.close()

# ===============================
# 6. GUARDAR EVENT LOG
# ===============================
events_df = pd.DataFrame(events).sort_values(["case_id", "timestamp"])
events_df.to_csv("data/processed/event_log.csv", index=False)

print("✅ Processamento concluído!")
print(f"Eventos criados: {len(events_df)}")
print("Ficheiro guardado em: data/processed/event_log.csv")


IndentationError: unindent does not match any outer indentation level (<tokenize>, line 124)

SECOND PIPELINE - THE ONE TO BE TESTED AND WHERE I CAN CHANGE THINGS AND TRY NEW CODE

In [37]:
import os
import pandas as pd
import matplotlib.pyplot as plt
from datetime import timedelta

# ===============================
# 1. CRIAR DIRETORIAS
# ===============================
base_dir = "caseid_meal"
dirs = ["graphs", "processed"]
for d in dirs:
    os.makedirs(os.path.join(base_dir, d), exist_ok=True)

# ===============================
# 2. CARREGAR DATASETS 
# ===============================

import pandas as pd

glucose = pd.read_csv('minha_pasta/sharpic-ManchesterCSCoordinatedDiabetesStudy-fdbd74f/Glucose Data/UoMGlucose2301.csv')
meals   = pd.read_csv('minha_pasta/sharpic-ManchesterCSCoordinatedDiabetesStudy-fdbd74f/Nutrition Data/UoMNutrition2301.csv')
bolus   = pd.read_csv('minha_pasta/sharpic-ManchesterCSCoordinatedDiabetesStudy-fdbd74f/Insulin Data/Bolus Data/UoMBolus2301.csv')

# Renomear colunas de tempo para um nome comum
glucose.rename(columns={'bg_ts': 'timestamp'}, inplace=True)
meals.rename(columns={'meal_ts': 'timestamp'}, inplace=True)
bolus.rename(columns={'bolus_ts': 'timestamp'}, inplace=True)

# Converter para datetime (formato DD/MM/YYYY)
glucose["timestamp"] = pd.to_datetime(glucose["timestamp"], dayfirst=True, errors='coerce')
meals["timestamp"] = pd.to_datetime(meals["timestamp"], dayfirst=True, errors='coerce')
bolus["timestamp"] = pd.to_datetime(bolus["timestamp"], dayfirst=True, errors='coerce')

# Validação opcional — ver se há datas inválidas
for name, df in [("value", glucose), ("meal_type", meals), ("bolus_dose", bolus)]:
    invalid = df["timestamp"].isna().sum()
    if invalid > 0:
        print(f"{invalid} timestamps inválidos encontrados em {name} (convertidos em NaT)")


# ===============================
# 2.5 CRIAR IDS ÚNICOS
# ===============================

glucose["id"] = range(1, len(glucose) + 1)
meals["id"]   = range(1, len(meals) + 1)
bolus["id"]   = range(1, len(bolus) + 1)


# ===============================
# 3. FUNÇÕES AUXILIARES
# ===============================

def get_next_meal(current_meal, all_meals):
    """Devolve a refeição seguinte."""
    later_meals = all_meals[all_meals["timestamp"] > current_meal["timestamp"]]
    if later_meals.empty:
        return None
    return later_meals.iloc[0]

def find_peak(glucose_window, start_time, baseline, hours_post_meal=3):
    """
    - Procura o valor máximo até 3h depois do início da refeição.
    - Se a glicose "voltar ao baseline" muito cedo (<30 min), ignora e considera 34h completas.

    Esta função identifica o pico máximo absoluto de glicose até 3 horas após a refeição, garantindo que o valor reflete a verdadeira resposta pós-prandial.
    Ignora retornos prematuros ao baseline (menores que 30 minutos) para evitar picos falsos causados por ruído ou leituras instáveis logo após a refeição.
    """

    # Filtrar dados a partir da refeição
    post_meal = glucose_window[glucose_window["timestamp"] >= start_time]

    # Procurar possível retorno ao baseline
    return_to_baseline = post_meal[post_meal["value"] <= baseline]

    # Determinar fim da janela de análise
    if not return_to_baseline.empty:
        end_time = return_to_baseline.iloc[0]["timestamp"]
        # Se o retorno for cedo demais, ignora
        if (end_time - start_time) < timedelta(minutes=30):
            end_time = start_time + timedelta(hours=hours_post_meal)
    else:
        end_time = start_time + timedelta(hours=hours_post_meal)

    # Recortar o período pós-prandial até 3h (ou até retorno válido)
    window = post_meal[
        (post_meal["timestamp"] >= start_time) &
        (post_meal["timestamp"] <= end_time)
    ]

    if window.empty:
        return None, None

    # Procurar o máximo absoluto nesse período
    idx = window["value"].idxmax()
    return window.loc[idx, "timestamp"], window.loc[idx, "value"], end_time



def find_recovery(glucose_df, peak_time, baseline, end_time):
    """
    Procura o primeiro timestamp após peak_time em que glucose <= baseline,
    até end_time. Se não existir, devolve None.
    """
    if peak_time is None:
        return None
    post_peak = glucose_df[
        (glucose_df["timestamp"] > peak_time) & 
        (glucose_df["timestamp"] <= end_time)
    ]
    post_peak_below = post_peak[post_peak["value"] <= baseline]
    if post_peak_below.empty:
        return None
    return post_peak_below.iloc[0]["timestamp"]

# ===============================
# 4. CRIAR EVENTOS COMPORTAMENTAIS
# ===============================
events = []

for _, meal in meals.iterrows():
    start_time = meal["timestamp"]
    end_time = start_time + timedelta(hours=3)
    
    next_meal = get_next_meal(meal, meals)
    if next_meal is not None and next_meal["timestamp"] < end_time:
        end_time = next_meal["timestamp"]
    
    glucose_window = glucose[(glucose["timestamp"] >= start_time) & (glucose["timestamp"] <= end_time)]
    if glucose_window.empty:
        continue

    # Baseline = average glucose in the 1h window before each meal
    glucose_before = glucose[(glucose["timestamp"] >= start_time - timedelta(minutes=60)) &
                             (glucose["timestamp"] < start_time)]
    baseline = glucose_before["value"].mean() if not glucose_before.empty else glucose_window.iloc[0]["value"]

    # Evento: Meal
    events.append({
        "case_id": meal["id"],
        "event_type": "Meal",
        "timestamp": start_time,
        "value": None
    })

    # Evento: Bolus (se aplicável)
    bolus_for_meal = bolus[(bolus["timestamp"] >= start_time - timedelta(minutes=30)) &
                           (bolus["timestamp"] <= start_time + timedelta(minutes=180)) & (bolus["bolus_dose"] > 0)]
    for _, b in bolus_for_meal.iterrows():
        events.append({
            "case_id": meal["id"],
            "event_type": "BolusInsulin",
            "timestamp": b["timestamp"],
            "value": b.get("bolus_dose", None)
        })

    # Evento: Pico de glicose
    peak_time, peak_value, computed_end_time = find_peak(glucose_window, start_time, baseline)
    if peak_time is not None:
        events.append({
            "case_id": meal["id"],
            "event_type": "PPGR_peak",
            "timestamp": peak_time,
            "value": peak_value
        })

    # Evento: Retorno ao normal
    recovery_time = find_recovery(glucose, peak_time, baseline, computed_end_time)
    if recovery_time:
        events.append({
            "case_id": meal["id"],
            "event_type": "PPGR_recovery",
            "timestamp": recovery_time,
            "value": baseline
        })

    # ===============================
    # 5. GERAR GRÁFICO
    # ===============================
    plt.figure(figsize=(10,5))
    
    # Plot da glicose
    plt.plot(glucose_window["timestamp"], glucose_window["value"], '-o', label="Glucose", zorder=1)
    
    # Meal
    plt.axvline(start_time, color='green', linestyle='--', label='Meal', zorder=2)
    
    # Peak (triângulo por cima da linha + valor do pico)
    plt.scatter(peak_time, peak_value, color='red', marker='^', s=100, label='Peak', zorder=3)
    plt.text(peak_time, peak_value + 0.2, f"{peak_value:.1f}", color='red', ha='center', zorder=4)
    
    # Return to baseline
    if recovery_time and (peak_time is None or recovery_time > peak_time):
        plt.axvline(recovery_time, color='orange', linestyle='--', label='Recovery', zorder=2)

    # Baseline
    plt.axhline(baseline, color='gray', linestyle=':', label=f'Baseline ({baseline:.1f})', zorder=1)
    
    # Bolus insulin (como bolinhas azuis)
    bolus_for_meal = bolus[(bolus["timestamp"] >= start_time - timedelta(minutes=30)) &
                           (bolus["timestamp"] <= start_time + timedelta(minutes=180)) & (bolus["bolus_dose"] > 0)]
    for _, b in bolus_for_meal.iterrows():
        plt.scatter(b["timestamp"], baseline, color='blue', marker='o', s=80, label='Bolus Insulin', zorder=3)
        plt.text(b["timestamp"], baseline + 0.2, f"{b.get('bolus_dose', 0):.1f}", color='blue', ha='center', zorder=4)

    
    plt.xlabel("Time")
    plt.ylabel("Glucose (mmol/L)")
    plt.title(f"Postprandial Glucose Response - Meal {meal['id']}")
    
    # Evitar legendas duplicadas
    handles, labels = plt.gca().get_legend_handles_labels()
    by_label = dict(zip(labels, handles))
    plt.legend(by_label.values(), by_label.keys())
    
    plt.tight_layout()
    plt.savefig(f"caseid_meal/processed/meal_{meal['id']}_curve.png")
    plt.close() 

    plt.figure(figsize=(10,5))

    # Plot da glicose
    plt.plot(glucose_window["timestamp"], glucose_window["value"], '-o', label="Glucose", zorder=1)
    
    # Meal
    plt.axvline(start_time, color='green', linestyle='--', label='Meal', zorder=2)
    
    # Peak (triângulo por cima da linha + valor do pico)
    plt.scatter(peak_time, peak_value, color='red', marker='^', s=100, label='Peak', zorder=3)
    plt.text(peak_time, peak_value + 0.2, f"{peak_value:.1f}", color='red', ha='center', zorder=4)
    
    # Return to baseline
    if recovery_time:
        plt.axvline(recovery_time, color='orange', linestyle='--', label='Recovery', zorder=2)
    
    # Baseline
    plt.axhline(baseline, color='gray', linestyle=':', label=f'Baseline ({baseline:.1f})', zorder=1)
    
    # Bolus insulin
    # Escolhemos altura ligeiramente acima do baseline para não sobrepor a linha
    bolus_for_meal = bolus[(bolus["timestamp"] >= start_time - timedelta(minutes=15)) &
                           (bolus["timestamp"] <= start_time + timedelta(minutes=15))]
    for _, b in bolus_for_meal.iterrows():
        plt.scatter(b["timestamp"], baseline + 0.3, color='green', marker='o', s=80, label='Bolus Insulin', zorder=3)
        plt.text(b["timestamp"], baseline + 0.6, f"{b.get('bolus_dose', 0):.1f}", color='blue', ha='center', zorder=4)
    
    plt.xlabel("Time")
    plt.ylabel("Glucose (mmol/L)")
    plt.title(f"Postprandial glucose response - meal {meal['id']} at {meal['timestamp'].strftime('%d/%m/%Y %H:%M') if pd.notna(meal['timestamp']) else 'unknown time'}")

    
    # Evitar legendas duplicadas
    handles, labels = plt.gca().get_legend_handles_labels()
    by_label = dict(zip(labels, handles))
    plt.legend(by_label.values(), by_label.keys())
    
    plt.tight_layout()
    plt.savefig(f"data/processed/meal_{meal['id']}_curve.png")
    plt.close()


# ===============================
# 6. GUARDAR EVENT LOG
# ===============================
events_df = pd.DataFrame(events).sort_values(["case_id", "timestamp"])
events_df.to_csv("caseid_meal/processed/event_log.csv", index=False)

print("Processamento concluído!")
print(f"Eventos criados: {len(events_df)}")
print("Ficheiro guardado em: caseid_meal/processed/event_log.csv")


✅ Processamento concluído!
Eventos criados: 1288
Ficheiro guardado em: caseid_meal/processed/event_log.csv


CASE ID: DAY

In [40]:
import os
import pandas as pd
import matplotlib.pyplot as plt
from datetime import timedelta

# ===============================
# 1. CRIAR DIRETORIAS
# ===============================
base_dir = "caseid_day"
dirs = ["graphs", "processed"]
for d in dirs:
    os.makedirs(os.path.join(base_dir, d), exist_ok=True)

# ===============================
# 2. CARREGAR DATASETS 
# ===============================

glucose = pd.read_csv('minha_pasta/sharpic-ManchesterCSCoordinatedDiabetesStudy-fdbd74f/Glucose Data/UoMGlucose2301.csv')
meals   = pd.read_csv('minha_pasta/sharpic-ManchesterCSCoordinatedDiabetesStudy-fdbd74f/Nutrition Data/UoMNutrition2301.csv')
bolus   = pd.read_csv('minha_pasta/sharpic-ManchesterCSCoordinatedDiabetesStudy-fdbd74f/Insulin Data/Bolus Data/UoMBolus2301.csv')

# Renomear colunas de tempo para um nome comum
glucose.rename(columns={'bg_ts': 'timestamp'}, inplace=True)
meals.rename(columns={'meal_ts': 'timestamp'}, inplace=True)
bolus.rename(columns={'bolus_ts': 'timestamp'}, inplace=True)

# Converter para datetime
glucose["timestamp"] = pd.to_datetime(glucose["timestamp"], dayfirst=True, errors='coerce')
meals["timestamp"] = pd.to_datetime(meals["timestamp"], dayfirst=True, errors='coerce')
bolus["timestamp"] = pd.to_datetime(bolus["timestamp"], dayfirst=True, errors='coerce')

# Criar coluna com o dia (novo CASE ID)
glucose["date"] = glucose["timestamp"].dt.date
meals["date"]   = meals["timestamp"].dt.date
bolus["date"]   = bolus["timestamp"].dt.date

# Validar timestamps inválidos
for name, df in [("value", glucose), ("meal_type", meals), ("bolus_dose", bolus)]:
    invalid = df["timestamp"].isna().sum()
    if invalid > 0:
        print(f"⚠️ {invalid} timestamps inválidos encontrados em {name} (convertidos em NaT)")

# ===============================
# 3. FUNÇÕES AUXILIARES
# ===============================

def get_next_meal(current_meal, all_meals):
    later_meals = all_meals[all_meals["timestamp"] > current_meal["timestamp"]]
    if later_meals.empty:
        return None
    return later_meals.iloc[0]

def find_peak(glucose_window, start_time, baseline, hours_post_meal=3):
    post_meal = glucose_window[glucose_window["timestamp"] >= start_time]
    return_to_baseline = post_meal[post_meal["value"] <= baseline]

    if not return_to_baseline.empty:
        end_time = return_to_baseline.iloc[0]["timestamp"]
        if (end_time - start_time) < timedelta(minutes=30):
            end_time = start_time + timedelta(hours=hours_post_meal)
    else:
        end_time = start_time + timedelta(hours=hours_post_meal)

    window = post_meal[
        (post_meal["timestamp"] >= start_time) &
        (post_meal["timestamp"] <= end_time)
    ]

    if window.empty:
        return None, None, None

    idx = window["value"].idxmax()
    return window.loc[idx, "timestamp"], window.loc[idx, "value"], end_time

def find_recovery(glucose_df, peak_time, baseline, end_time):
    if peak_time is None:
        return None
    post_peak = glucose_df[
        (glucose_df["timestamp"] > peak_time) &
        (glucose_df["timestamp"] <= end_time)
    ]
    post_peak_below = post_peak[post_peak["value"] <= baseline]
    if post_peak_below.empty:
        return None
    return post_peak_below.iloc[0]["timestamp"]

# ===============================
# 4. CRIAR EVENTOS
# ===============================
events = []

for _, meal in meals.iterrows():
    start_time = meal["timestamp"]
    end_time = start_time + timedelta(hours=3)

    next_meal = get_next_meal(meal, meals)
    if next_meal is not None and next_meal["timestamp"] < end_time:
        end_time = next_meal["timestamp"]

    glucose_window = glucose[(glucose["timestamp"] >= start_time) &
                             (glucose["timestamp"] <= end_time)]
    if glucose_window.empty:
        continue

    glucose_before = glucose[(glucose["timestamp"] >= start_time - timedelta(minutes=60)) &
                             (glucose["timestamp"] < start_time)]
    baseline = glucose_before["value"].mean() if not glucose_before.empty else glucose_window.iloc[0]["value"]

    case_id = str(meal["date"])   # 🔥 NOVO CASE ID (dia)

    # Meal event
    events.append({
        "case_id": case_id,
        "event_type": "Meal",
        "timestamp": start_time,
        "value": None
    })

    # Bolus event
    bolus_for_meal = bolus[(bolus["timestamp"] >= start_time - timedelta(minutes=30)) &
                           (bolus["timestamp"] <= start_time + timedelta(minutes=180)) & 
                           (bolus["bolus_dose"] > 0)]
    for _, b in bolus_for_meal.iterrows():
        events.append({
            "case_id": case_id,
            "event_type": "BolusInsulin",
            "timestamp": b["timestamp"],
            "value": b.get("bolus_dose", None)
        })

    # Peak
    peak_time, peak_value, computed_end_time = find_peak(glucose_window, start_time, baseline)
    if peak_time is not None:
        events.append({
            "case_id": case_id,
            "event_type": "PPGR_peak",
            "timestamp": peak_time,
            "value": peak_value
        })

    # Recovery
    recovery_time = find_recovery(glucose, peak_time, baseline, computed_end_time)
    if recovery_time:
        events.append({
            "case_id": case_id,
            "event_type": "PPGR_recovery",
            "timestamp": recovery_time,
            "value": baseline
        })

# ===============================
# 5. GERAR GRÁFICOS POR DIA (24H)
# ===============================

# Obter todos os dias existentes nos dados de glucose
all_days = sorted(glucose["date"].unique())

for day in all_days:

    # Dados de glicose do dia inteiro
    day_glucose = glucose[glucose["date"] == day]

    if day_glucose.empty:
        continue

    # Refeições do dia
    day_meals = meals[meals["date"] == day]

    # Bolus do dia
    day_bolus = bolus[bolus["date"] == day]

    # Criar gráfico
    plt.figure(figsize=(14, 6))

    # — Glucose curve —
    plt.plot(day_glucose["timestamp"], day_glucose["value"], '-o', label="Glucose", zorder=1)

    # — Meals —
    for _, meal in day_meals.iterrows():
        plt.axvline(meal["timestamp"], color='green', linestyle='--', alpha=0.7, label='Meal')
    
    # — Bolus insulin —
    for _, b in day_bolus.iterrows():
        plt.scatter(b["timestamp"], day_glucose["value"].min() - 0.2,
                    color='blue', marker='o', s=80, label="Bolus")
        plt.text(b["timestamp"],
                 day_glucose["value"].min() - 0.5,
                 f"{b.get('bolus_dose',0):.1f}",
                 color="blue", ha="center")

    # — PPGR peaks —
    peaks_today = [ev for ev in events if ev["case_id"] == str(day) and ev["event_type"] == "PPGR_peak"]
    for pk in peaks_today:
        plt.scatter(pk["timestamp"], pk["value"], color='red', marker='^', s=120, label="PPGR_peak")
        plt.text(pk["timestamp"], pk["value"] + 0.2, f"{pk['value']:.1f}", color='red', ha='center')

    # — Recovery to baseline —
    recov_today = [ev for ev in events if ev["case_id"] == str(day) and ev["event_type"] == "PPGR_recovery"]
    for rc in recov_today:
        plt.axvline(rc["timestamp"], color='orange', linestyle='--', label="Recovery")

    plt.title(f"Daily Glucose Profile — {day}")
    plt.xlabel("Time")
    plt.ylabel("Glucose (mmol/L)")

    # Evitar duplicados na legenda
    handles, labels = plt.gca().get_legend_handles_labels()
    by_label = dict(zip(labels, handles))
    plt.legend(by_label.values(), by_label.keys())

    plt.tight_layout()
    plt.savefig(f"caseid_day/processed/day_{day}_curve.png")
    plt.close()


# ===============================
# 6. EXPORT EVENT LOG
# ===============================
events_df = pd.DataFrame(events).sort_values(["case_id", "timestamp"])
events_df.to_csv("caseid_day/processed/event_log.csv", index=False)

print("✅ Processamento concluído!")
print(f"Eventos criados: {len(events_df)}")
print("Ficheiro guardado em: caseid_day/processed/event_log.csv")


✅ Processamento concluído!
Eventos criados: 1288
Ficheiro guardado em: caseid_day/processed/event_log.csv


In [ ]:
CASE ID: MEAL WITH HYPERGLYCEMIA AND HYPOGLICEMIA FILTER

In [42]:
import os
import pandas as pd
import matplotlib.pyplot as plt
from datetime import timedelta

# ===============================
# 1. CRIAR DIRETORIAS
# ===============================
base_dir = "caseid_meal_HypoHyper"
dirs = ["graphs", "processed"]
for d in dirs:
    os.makedirs(os.path.join(base_dir, d), exist_ok=True)

# ===============================
# 2. CARREGAR DATASETS 
# ===============================

import pandas as pd

glucose = pd.read_csv('minha_pasta/sharpic-ManchesterCSCoordinatedDiabetesStudy-fdbd74f/Glucose Data/UoMGlucose2301.csv')
meals   = pd.read_csv('minha_pasta/sharpic-ManchesterCSCoordinatedDiabetesStudy-fdbd74f/Nutrition Data/UoMNutrition2301.csv')
bolus   = pd.read_csv('minha_pasta/sharpic-ManchesterCSCoordinatedDiabetesStudy-fdbd74f/Insulin Data/Bolus Data/UoMBolus2301.csv')

# Renomear colunas de tempo para um nome comum
glucose.rename(columns={'bg_ts': 'timestamp'}, inplace=True)
meals.rename(columns={'meal_ts': 'timestamp'}, inplace=True)
bolus.rename(columns={'bolus_ts': 'timestamp'}, inplace=True)

# Converter para datetime (formato DD/MM/YYYY)
glucose["timestamp"] = pd.to_datetime(glucose["timestamp"], dayfirst=True, errors='coerce')
meals["timestamp"] = pd.to_datetime(meals["timestamp"], dayfirst=True, errors='coerce')
bolus["timestamp"] = pd.to_datetime(bolus["timestamp"], dayfirst=True, errors='coerce')

# Validação opcional — ver se há datas inválidas
for name, df in [("value", glucose), ("meal_type", meals), ("bolus_dose", bolus)]:
    invalid = df["timestamp"].isna().sum()
    if invalid > 0:
        print(f"{invalid} timestamps inválidos encontrados em {name} (convertidos em NaT)")


# ===============================
# 2.5 CRIAR IDS ÚNICOS
# ===============================

glucose["id"] = range(1, len(glucose) + 1)
meals["id"]   = range(1, len(meals) + 1)
bolus["id"]   = range(1, len(bolus) + 1)


# ===============================
# 3. FUNÇÕES AUXILIARES
# ===============================

def get_next_meal(current_meal, all_meals):
    """Devolve a refeição seguinte."""
    later_meals = all_meals[all_meals["timestamp"] > current_meal["timestamp"]]
    if later_meals.empty:
        return None
    return later_meals.iloc[0]

def find_peak(glucose_window, start_time, baseline, hours_post_meal=3):
    """
    - Procura o valor máximo até 3h depois do início da refeição.
    - Se a glicose "voltar ao baseline" muito cedo (<30 min), ignora e considera 34h completas.

    Esta função identifica o pico máximo absoluto de glicose até 3 horas após a refeição, garantindo que o valor reflete a verdadeira resposta pós-prandial.
    Ignora retornos prematuros ao baseline (menores que 30 minutos) para evitar picos falsos causados por ruído ou leituras instáveis logo após a refeição.
    """

    # Filtrar dados a partir da refeição
    post_meal = glucose_window[glucose_window["timestamp"] >= start_time]

    # Procurar possível retorno ao baseline
    return_to_baseline = post_meal[post_meal["value"] <= baseline]

    # Determinar fim da janela de análise
    if not return_to_baseline.empty:
        end_time = return_to_baseline.iloc[0]["timestamp"]
        # Se o retorno for cedo demais, ignora
        if (end_time - start_time) < timedelta(minutes=30):
            end_time = start_time + timedelta(hours=hours_post_meal)
    else:
        end_time = start_time + timedelta(hours=hours_post_meal)

    # Recortar o período pós-prandial até 3h (ou até retorno válido)
    window = post_meal[
        (post_meal["timestamp"] >= start_time) &
        (post_meal["timestamp"] <= end_time)
    ]

    if window.empty:
        return None, None

    # Procurar o máximo absoluto nesse período
    idx = window["value"].idxmax()
    return window.loc[idx, "timestamp"], window.loc[idx, "value"], end_time



def find_recovery(glucose_df, peak_time, baseline, end_time):
    """
    Procura o primeiro timestamp após peak_time em que glucose <= baseline,
    até end_time. Se não existir, devolve None.
    """
    if peak_time is None:
        return None
    post_peak = glucose_df[
        (glucose_df["timestamp"] > peak_time) & 
        (glucose_df["timestamp"] <= end_time)
    ]
    post_peak_below = post_peak[post_peak["value"] <= baseline]
    if post_peak_below.empty:
        return None
    return post_peak_below.iloc[0]["timestamp"]

# ===============================
# 4. CRIAR EVENTOS COMPORTAMENTAIS
# ===============================
events = []

for _, meal in meals.iterrows():
    start_time = meal["timestamp"]
    end_time = start_time + timedelta(hours=3)
    
    next_meal = get_next_meal(meal, meals)
    if next_meal is not None and next_meal["timestamp"] < end_time:
        end_time = next_meal["timestamp"]
    
    glucose_window = glucose[(glucose["timestamp"] >= start_time) & (glucose["timestamp"] <= end_time)]
    if glucose_window.empty:
        continue

    # Baseline = average glucose in the 1h window before each meal
    glucose_before = glucose[(glucose["timestamp"] >= start_time - timedelta(minutes=60)) &
                             (glucose["timestamp"] < start_time)]
    baseline = glucose_before["value"].mean() if not glucose_before.empty else glucose_window.iloc[0]["value"]

    # Evento: Meal
    events.append({
        "case_id": meal["id"],
        "event_type": "Meal",
        "timestamp": start_time,
        "value": None
    })

    # Evento: Bolus (se aplicável)
    bolus_for_meal = bolus[(bolus["timestamp"] >= start_time - timedelta(minutes=30)) &
                           (bolus["timestamp"] <= start_time + timedelta(minutes=180)) & (bolus["bolus_dose"] > 0)]
    for _, b in bolus_for_meal.iterrows():
        events.append({
            "case_id": meal["id"],
            "event_type": "BolusInsulin",
            "timestamp": b["timestamp"],
            "value": b.get("bolus_dose", None)
        })

    # Evento: Pico de glicose
    peak_time, peak_value, computed_end_time = find_peak(glucose_window, start_time, baseline)
    if peak_time is not None:
        events.append({
            "case_id": meal["id"],
            "event_type": "PPGR_peak",
            "timestamp": peak_time,
            "value": peak_value
        })

    # Evento: Retorno ao normal
    recovery_time = find_recovery(glucose, peak_time, baseline, computed_end_time)
    if recovery_time:
        events.append({
            "case_id": meal["id"],
            "event_type": "PPGR_recovery",
            "timestamp": recovery_time,
            "value": baseline
        })


    # ===============================
    # 4.5 — DETEÇÃO DE HYPER/HYPOGLYCEMIA
    # ===============================

    # Recorta os dados de glucose apenas dentro do case (meal até next meal)
    glucose_case = glucose[(glucose["timestamp"] >= start_time) &
                           (glucose["timestamp"] <= end_time)].sort_values("timestamp")

    hyper_threshold = 7.8
    hypo_threshold = 3.9

    in_hyper = False
    in_hypo = False

    for _, row in glucose_case.iterrows():
        g = row["value"]
        t = row["timestamp"]

        # ---------- Hyperglycemia ----------
        if g > hyper_threshold and not in_hyper:
            # Start event
            events.append({
                "case_id": meal["id"],
                "event_type": "HyperglycemiaStart",
                "timestamp": t,
                "value": g
            })
            in_hyper = True

        if g <= hyper_threshold and in_hyper:
            events.append({
                "case_id": meal["id"],
                "event_type": "HyperglycemiaEnd",
                "timestamp": t,
                "value": g
            })
            in_hyper = False

        # ---------- Hypoglycemia ----------
        if g < hypo_threshold and not in_hypo:
            events.append({
                "case_id": meal["id"],
                "event_type": "HypoglycemiaStart",
                "timestamp": t,
                "value": g
            })
            in_hypo = True

        if g >= hypo_threshold and in_hypo:
            events.append({
                "case_id": meal["id"],
                "event_type": "HypoglycemiaEnd",
                "timestamp": t,
                "value": g
            })
            in_hypo = False


    # ===============================
    # 5. GERAR GRÁFICO
    # ===============================
    plt.figure(figsize=(10,5))
    
    # Plot da glicose
    plt.plot(glucose_window["timestamp"], glucose_window["value"], '-o', label="Glucose", zorder=1)
    
    # Meal
    plt.axvline(start_time, color='green', linestyle='--', label='Meal', zorder=2)
    
    # Peak (triângulo por cima da linha + valor do pico)
    plt.scatter(peak_time, peak_value, color='red', marker='^', s=100, label='Peak', zorder=3)
    plt.text(peak_time, peak_value + 0.2, f"{peak_value:.1f}", color='red', ha='center', zorder=4)
    
    # Return to baseline
    if recovery_time and (peak_time is None or recovery_time > peak_time):
        plt.axvline(recovery_time, color='orange', linestyle='--', label='Recovery', zorder=2)

    # Baseline
    plt.axhline(baseline, color='gray', linestyle=':', label=f'Baseline ({baseline:.1f})', zorder=1)
    
    # Bolus insulin (como bolinhas azuis)
    bolus_for_meal = bolus[(bolus["timestamp"] >= start_time - timedelta(minutes=30)) &
                           (bolus["timestamp"] <= start_time + timedelta(minutes=180)) & (bolus["bolus_dose"] > 0)]
    for _, b in bolus_for_meal.iterrows():
        plt.scatter(b["timestamp"], baseline, color='blue', marker='o', s=80, label='Bolus Insulin', zorder=3)
        plt.text(b["timestamp"], baseline + 0.2, f"{b.get('bolus_dose', 0):.1f}", color='blue', ha='center', zorder=4)

    
    plt.xlabel("Time")
    plt.ylabel("Glucose (mmol/L)")
    plt.title(f"Postprandial Glucose Response - Meal {meal['id']}")
    
    # Evitar legendas duplicadas
    handles, labels = plt.gca().get_legend_handles_labels()
    by_label = dict(zip(labels, handles))
    plt.legend(by_label.values(), by_label.keys())
    
    plt.tight_layout()
    plt.savefig(f"caseid_meal_HypoHyper/processed/meal_{meal['id']}_curve.png")
    plt.close() 

    plt.figure(figsize=(10,5))

    # Plot da glicose
    gw = glucose_window.sort_values("timestamp")
    hyper_threshold = 7.8
    hypo_threshold  = 3.9
    
    ts = gw["timestamp"].values
    vs = gw["value"].values
    
    for i in range(len(gw) - 1):
        t0, t1 = ts[i], ts[i+1]
        v0, v1 = vs[i], vs[i+1]
    
        # Verificar condição do segmento
        if v0 > hyper_threshold or v1 > hyper_threshold or v0 < hypo_threshold or v1 < hypo_threshold:
            color = "red"
        else:
            color = "blue"
    
        plt.plot([t0, t1], [v0, v1], color=color, linewidth=2)
    
    # Meal
    plt.axvline(start_time, color='green', linestyle='--', label='Meal', zorder=2)
    
    # Peak (triângulo por cima da linha + valor do pico)
    plt.scatter(peak_time, peak_value, color='red', marker='^', s=100, label='Peak', zorder=3)
    plt.text(peak_time, peak_value + 0.2, f"{peak_value:.1f}", color='red', ha='center', zorder=4)
    
    # Return to baseline
    if recovery_time:
        plt.axvline(recovery_time, color='orange', linestyle='--', label='Recovery', zorder=2)
    
    # Baseline
    plt.axhline(baseline, color='gray', linestyle=':', label=f'Baseline ({baseline:.1f})', zorder=1)
    
    # Bolus insulin
    # Escolhemos altura ligeiramente acima do baseline para não sobrepor a linha
    bolus_for_meal = bolus[(bolus["timestamp"] >= start_time - timedelta(minutes=15)) &
                           (bolus["timestamp"] <= start_time + timedelta(minutes=15))]
    for _, b in bolus_for_meal.iterrows():
        plt.scatter(b["timestamp"], baseline + 0.3, color='green', marker='o', s=80, label='Bolus Insulin', zorder=3)
        plt.text(b["timestamp"], baseline + 0.6, f"{b.get('bolus_dose', 0):.1f}", color='blue', ha='center', zorder=4)
    
    plt.xlabel("Time")
    plt.ylabel("Glucose (mmol/L)")
    plt.title(f"Postprandial glucose response - meal {meal['id']} at {meal['timestamp'].strftime('%d/%m/%Y %H:%M') if pd.notna(meal['timestamp']) else 'unknown time'}")

    
    # Evitar legendas duplicadas
    handles, labels = plt.gca().get_legend_handles_labels()
    by_label = dict(zip(labels, handles))
    plt.legend(by_label.values(), by_label.keys())
    
    plt.tight_layout()
    plt.savefig(f"caseid_meal_HypoHyper/processed/meal_{meal['id']}_curve.png")
    plt.close()


# ===============================
# 6. GUARDAR EVENT LOG
# ===============================
events_df = pd.DataFrame(events).sort_values(["case_id", "timestamp"])
events_df.to_csv("caseid_meal_HypoHyper/processed/event_log.csv", index=False)

print("Processamento concluído!")
print(f"Eventos criados: {len(events_df)}")
print("Ficheiro guardado em: caseid_meal_HypoHyper/processed/event_log.csv")


Processamento concluído!
Eventos criados: 1816
Ficheiro guardado em: caseid_meal_HypoHyper/processed/event_log.csv


In [ ]:
CASE ID: DAY WITH HYPERGLYCEMIA AND HYPOGLICEMIA FILTER

In [43]:
import os
import pandas as pd
import matplotlib.pyplot as plt
from datetime import timedelta

# ===============================
# 1. CRIAR DIRETORIAS
# ===============================
base_dir = "caseid_day_HypoHyper"
dirs = ["graphs", "processed"]
for d in dirs:
    os.makedirs(os.path.join(base_dir, d), exist_ok=True)

# ===============================
# 2. CARREGAR DATASETS 
# ===============================

glucose = pd.read_csv('minha_pasta/sharpic-ManchesterCSCoordinatedDiabetesStudy-fdbd74f/Glucose Data/UoMGlucose2301.csv')
meals   = pd.read_csv('minha_pasta/sharpic-ManchesterCSCoordinatedDiabetesStudy-fdbd74f/Nutrition Data/UoMNutrition2301.csv')
bolus   = pd.read_csv('minha_pasta/sharpic-ManchesterCSCoordinatedDiabetesStudy-fdbd74f/Insulin Data/Bolus Data/UoMBolus2301.csv')

# Renomear colunas de tempo para um nome comum
glucose.rename(columns={'bg_ts': 'timestamp'}, inplace=True)
meals.rename(columns={'meal_ts': 'timestamp'}, inplace=True)
bolus.rename(columns={'bolus_ts': 'timestamp'}, inplace=True)

# Converter para datetime
glucose["timestamp"] = pd.to_datetime(glucose["timestamp"], dayfirst=True, errors='coerce')
meals["timestamp"] = pd.to_datetime(meals["timestamp"], dayfirst=True, errors='coerce')
bolus["timestamp"] = pd.to_datetime(bolus["timestamp"], dayfirst=True, errors='coerce')

# Criar coluna com o dia (novo CASE ID)
glucose["date"] = glucose["timestamp"].dt.date
meals["date"]   = meals["timestamp"].dt.date
bolus["date"]   = bolus["timestamp"].dt.date

# Validar timestamps inválidos
for name, df in [("value", glucose), ("meal_type", meals), ("bolus_dose", bolus)]:
    invalid = df["timestamp"].isna().sum()
    if invalid > 0:
        print(f"⚠️ {invalid} timestamps inválidos encontrados em {name} (convertidos em NaT)")

# ===============================
# 3. FUNÇÕES AUXILIARES
# ===============================

def get_next_meal(current_meal, all_meals):
    later_meals = all_meals[all_meals["timestamp"] > current_meal["timestamp"]]
    if later_meals.empty:
        return None
    return later_meals.iloc[0]

def find_peak(glucose_window, start_time, baseline, hours_post_meal=3):
    post_meal = glucose_window[glucose_window["timestamp"] >= start_time]
    return_to_baseline = post_meal[post_meal["value"] <= baseline]

    if not return_to_baseline.empty:
        end_time = return_to_baseline.iloc[0]["timestamp"]
        if (end_time - start_time) < timedelta(minutes=30):
            end_time = start_time + timedelta(hours=hours_post_meal)
    else:
        end_time = start_time + timedelta(hours=hours_post_meal)

    window = post_meal[
        (post_meal["timestamp"] >= start_time) &
        (post_meal["timestamp"] <= end_time)
    ]

    if window.empty:
        return None, None, None

    idx = window["value"].idxmax()
    return window.loc[idx, "timestamp"], window.loc[idx, "value"], end_time

def find_recovery(glucose_df, peak_time, baseline, end_time):
    if peak_time is None:
        return None
    post_peak = glucose_df[
        (glucose_df["timestamp"] > peak_time) &
        (glucose_df["timestamp"] <= end_time)
    ]
    post_peak_below = post_peak[post_peak["value"] <= baseline]
    if post_peak_below.empty:
        return None
    return post_peak_below.iloc[0]["timestamp"]

# ===============================
# 4. CRIAR EVENTOS
# ===============================
events = []

hyper_threshold = 7.8
hypo_threshold  = 3.9


for _, meal in meals.iterrows():
    start_time = meal["timestamp"]
    end_time = start_time + timedelta(hours=3)

    next_meal = get_next_meal(meal, meals)
    if next_meal is not None and next_meal["timestamp"] < end_time:
        end_time = next_meal["timestamp"]

    glucose_window = glucose[(glucose["timestamp"] >= start_time) &
                             (glucose["timestamp"] <= end_time)]
    if glucose_window.empty:
        continue

    glucose_before = glucose[(glucose["timestamp"] >= start_time - timedelta(minutes=60)) &
                             (glucose["timestamp"] < start_time)]
    baseline = glucose_before["value"].mean() if not glucose_before.empty else glucose_window.iloc[0]["value"]

    case_id = str(meal["date"])   # 🔥 CASE ID = DIA

    # Meal event
    events.append({
        "case_id": case_id,
        "event_type": "Meal",
        "timestamp": start_time,
        "value": None
    })

    # Bolus event
    bolus_for_meal = bolus[(bolus["timestamp"] >= start_time - timedelta(minutes=30)) &
                           (bolus["timestamp"] <= start_time + timedelta(minutes=180)) & 
                           (bolus["bolus_dose"] > 0)]
    for _, b in bolus_for_meal.iterrows():
        events.append({
            "case_id": case_id,
            "event_type": "BolusInsulin",
            "timestamp": b["timestamp"],
            "value": b.get("bolus_dose", None)
        })

    # Peak
    peak_time, peak_value, computed_end_time = find_peak(glucose_window, start_time, baseline)
    if peak_time is not None:
        events.append({
            "case_id": case_id,
            "event_type": "PPGR_peak",
            "timestamp": peak_time,
            "value": peak_value
        })

    # Recovery
    recovery_time = find_recovery(glucose, peak_time, baseline, computed_end_time)
    if recovery_time:
        events.append({
            "case_id": case_id,
            "event_type": "PPGR_recovery",
            "timestamp": recovery_time,
            "value": baseline
        })


# ===============================
# 4.5 ADIÇÃO — DETEÇÃO DIÁRIA DE HYPER / HYPO
# ===============================

for day, day_glucose in glucose.groupby("date"):

    day_glucose = day_glucose.sort_values("timestamp")
    case_id = str(day)

    in_hyper = False
    in_hypo  = False

    for _, row in day_glucose.iterrows():
        g = row["value"]
        t = row["timestamp"]

        # ---------- Hyperglycemia ----------
        if g > hyper_threshold and not in_hyper:
            events.append({
                "case_id": case_id,
                "event_type": "HyperglycemiaStart",
                "timestamp": t,
                "value": g
            })
            in_hyper = True

        if g <= hyper_threshold and in_hyper:
            events.append({
                "case_id": case_id,
                "event_type": "HyperglycemiaEnd",
                "timestamp": t,
                "value": g
            })
            in_hyper = False

        # ---------- Hypoglycemia ----------
        if g < hypo_threshold and not in_hypo:
            events.append({
                "case_id": case_id,
                "event_type": "HypoglycemiaStart",
                "timestamp": t,
                "value": g
            })
            in_hypo = True
        
        if g >= hypo_threshold and in_hypo:
            events.append({
                "case_id": case_id,
                "event_type": "HypoglycemiaEnd",
                "timestamp": t,
                "value": g
            })
            in_hypo = False


# ===============================
# 5. GERAR GRÁFICOS POR DIA (24H)
# ===============================

# Obter todos os dias existentes nos dados de glucose
all_days = sorted(glucose["date"].unique())

for day in all_days:

    # Dados de glicose do dia inteiro
    day_glucose = glucose[glucose["date"] == day]

    if day_glucose.empty:
        continue

    # Refeições do dia
    day_meals = meals[meals["date"] == day]

    # Bolus do dia
    day_bolus = bolus[bolus["date"] == day]

    # Criar gráfico
    plt.figure(figsize=(14, 6))

    # — Glucose curve —
    plt.plot(day_glucose["timestamp"], day_glucose["value"], '-o', label="Glucose", zorder=1)

    # — Meals —
    for _, meal in day_meals.iterrows():
        plt.axvline(meal["timestamp"], color='green', linestyle='--', alpha=0.7, label='Meal')
    
    # — Bolus insulin —
    for _, b in day_bolus.iterrows():
        plt.scatter(b["timestamp"], day_glucose["value"].min() - 0.2,
                    color='blue', marker='o', s=80, label="Bolus")
        plt.text(b["timestamp"],
                 day_glucose["value"].min() - 0.5,
                 f"{b.get('bolus_dose',0):.1f}",
                 color="blue", ha="center")

    # — PPGR peaks —
    peaks_today = [ev for ev in events if ev["case_id"] == str(day) and ev["event_type"] == "PPGR_peak"]
    for pk in peaks_today:
        plt.scatter(pk["timestamp"], pk["value"], color='red', marker='^', s=120, label="PPGR_peak")
        plt.text(pk["timestamp"], pk["value"] + 0.2, f"{pk['value']:.1f}", color='red', ha='center')

    # — Recovery to baseline —
    recov_today = [ev for ev in events if ev["case_id"] == str(day) and ev["event_type"] == "PPGR_recovery"]
    for rc in recov_today:
        plt.axvline(rc["timestamp"], color='orange', linestyle='--', label="Recovery")

    plt.title(f"Daily Glucose Profile — {day}")
    plt.xlabel("Time")
    plt.ylabel("Glucose (mmol/L)")

    # Evitar duplicados na legenda
    handles, labels = plt.gca().get_legend_handles_labels()
    by_label = dict(zip(labels, handles))
    plt.legend(by_label.values(), by_label.keys())

    plt.tight_layout()
    plt.savefig(f"caseid_day_HypoHyper/processed/day_{day}_curve.png")
    plt.close()


# ===============================
# 6. EXPORT EVENT LOG
# ===============================
events_df = pd.DataFrame(events).sort_values(["case_id", "timestamp"])
events_df.to_csv("caseid_day_HypoHyper/processed/event_log.csv", index=False)

print("✅ Processamento concluído!")
print(f"Eventos criados: {len(events_df)}")
print("Ficheiro guardado em: caseid_day_HypoHyper/processed/event_log.csv")


✅ Processamento concluído!
Eventos criados: 2482
Ficheiro guardado em: caseid_day_HypoHyper/processed/event_log.csv
